# 🇻🇳 ViMind 4.0: Bước Nhảy Nhận Thức (The Cognitive Sovereign)
### (Mixture-of-Experts 198M Top-2 Routing, Targeted Pre-training, CoT SFT, DPO & Agent RL)

Notebook này thiết lập chu trình khép kín tối tân nhất cho **ViMind 4.0** trên **Kaggle GPU (Tesla T4 16GB VRAM)**:
1. **Pha 1: Targeted Pre-training (MoE 198M - Top-2 Routing):** Tiền huấn luyện trên Wikipedia tiếng Việt, Sách giáo khoa Toán/Khoa học, và Tri thức nền tảng Việt Nam. Xây dựng bản sắc và vốn tri thức thực tế trước khi SFT.
2. **Pha 2: SFT 4.0 (CoT & Factual Grounding):** Nạp từ Base Model vừa tiền huấn luyện, học 52,000+ mẫu hội thoại tiếng Việt kết hợp chuỗi suy nghĩ `<think>` và Factual Grounding.
3. **Pha 3: Anti-Refusal DPO:** Căn chỉnh sở thích, loại bỏ hoàn toàn hiện tượng từ chối máy móc và ảo giác.
4. **Pha 4: Agentic RL (GRPO Tool-Use):** Huấn luyện kỹ năng kích hoạt công cụ giải toán, tra thời tiết, thời gian qua `<tool_call>`.
5. **Pha 5: Anti-Degeneration Inference & Packaging:** Đóng gói SafeTensors và kiểm thử suy luận với Repetition Penalty 1.2 & N-gram blocking.

In [ ]:
# 1. Đồng bộ mã nguồn ViMind 4.0 từ GitHub
!rm -rf /kaggle/working/vimind
!git clone https://github.com/WuKong0601/ViMind.git /kaggle/working/vimind
%cd /kaggle/working/vimind


In [ ]:
# 2. Cài đặt các gói phụ thuộc (PyTorch, Transformers, BitsAndBytes, SafeTensors, FastAPI)
!pip install -r requirements.txt
!pip install -q bitsandbytes accelerate fastapi uvicorn requests


In [ ]:
# 3. Kiểm tra phần cứng GPU Tesla T4 & môi trường CUDA
!nvidia-smi
import torch
print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'GPU Device: {gpu_name} ({total_mem:.1f} GB VRAM)')


In [ ]:
# 4. Chạy Dry-Run kiểm thử toàn diện 5 thành phần ViMind 4.0
# (Kiểm tra MoE Top-2 Routing, Anti-Degeneration Decoding, Transformers 5.0+ Compatibility)
!python trainer/test_4.0_dry_run.py


In [ ]:
# 5. [GIAI ĐOẠN 1A: TỔNG HỢP TRI THỨC VÀNG & GOLDEN DENSE REASONING CORE]
# 1. Sinh tập dữ liệu Golden Dense Core (10k mẫu tri thức thuần khiết, 0% Alpaca dịch máy rác)
import os
print('💎 Đang biên dịch tập dữ liệu Golden Dense Core (Toán <think>, Địa lý VN, Khoa học, Tool)...')
!python data_pipeline/build_dense_reasoning_core.py dataset/dense_core_vi.jsonl
!cp dataset/dense_core_vi.jsonl dataset/sft_vi_v4.jsonl

# 2. Chuẩn bị tập dữ liệu tiền huấn luyện nếu chưa có
if not os.path.exists('dataset/pretrain_vi_v4.jsonl') or os.path.getsize('dataset/pretrain_vi_v4.jsonl') < 100000:
    print('📚 Biên dịch tập dữ liệu tiền huấn luyện Wikipedia...')
    !python data_pipeline/prepare_pretrain_4.0.py --max_wiki 50000 --max_math 15000
else:
    print('⚡ Đã có sẵn dataset/pretrain_vi_v4.jsonl!')


In [ ]:
# 6. [GIAI ĐOẠN 1B: TARGETED PRE-TRAINING (MoE 198M - Top-2 Routing)]
# Tiền huấn luyện từ con số 0 trên Wikipedia tiếng Việt và tri thức chuẩn
!python -u trainer/pretrain.py \
    --data_path dataset/pretrain_vi_v4.jsonl \
    --tokenizer_dir model \
    --save_dir out/pretrain \
    --save_weight vimind_4.0_base \
    --model_size 64m \
    --use_moe \
    --num_experts 4 \
    --num_experts_per_tok 2 \
    --batch_size 8 \
    --accumulation_steps 16 \
    --max_seq_len 512 \
    --gradient_checkpointing \
    --epochs 1 \
    --max_steps 3000 \
    --learning_rate 4e-4 \
    --dtype float16 \
    --log_interval 50 \
    --save_interval 1000

# Dọn dẹp checkpoints trung gian để bảo toàn dung lượng ổ đĩa Kaggle
!rm -rf out/pretrain/*_step_*
!ls -lh out/pretrain


In [ ]:
# 7. [GIAI ĐOẠN 2: HUẤN LUYỆN SFT NATIVE (Học tập trung 4 Epochs trên Golden Core)]
# Nhồi trực tiếp 10k mẫu Golden Dense Core để ghim chặt tri thức vào MoE 4 experts
import os
base_checkpoint = 'out/pretrain/vimind_4.0_base_final' if os.path.exists('out/pretrain/vimind_4.0_base_final') else ('out/pretrain/vimind_4.0_base.pth' if os.path.exists('out/pretrain/vimind_4.0_base.pth') else 'none')
!python -u trainer/train_sft.py \
    --data_path dataset/dense_core_vi.jsonl \
    --from_pretrained {base_checkpoint} \
    --tokenizer_dir model \
    --save_dir out/sft_moe \
    --save_weight vimind_4.0_moe \
    --use_moe \
    --num_experts 4 \
    --num_experts_per_tok 2 \
    --batch_size 8 \
    --accumulation_steps 8 \
    --max_seq_len 512 \
    --gradient_checkpointing \
    --epochs 4 \
    --max_steps 2500 \
    --learning_rate 2e-4 \
    --dtype float16 \
    --log_interval 50 \
    --save_interval 1000

# Dọn dẹp checkpoints trung gian SFT
!rm -rf out/sft_moe/*_step_*
!ls -lh out/sft_moe


In [ ]:
# 8. [TRACK 2: HUẤN LUYỆN VIMIND 4.0 PRO (0.5B Foundation Model Alignment)]
# Chuyển giao tri thức nhân loại (18k tỷ tokens) sang tiếng Việt và chuỗi tư duy <think> của ViMind
!python -u trainer/train_pro_sft.py \
    --model_name_or_path Qwen/Qwen2.5-0.5B-Instruct \
    --data_path dataset/dense_core_vi.jsonl \
    --output_dir /kaggle/working/vimind_4.0_pro_final \
    --epochs 2 \
    --max_steps 500 \
    --batch_size 4 \
    --accumulation_steps 8 \
    --learning_rate 2e-5 \
    --fp16

!ls -lh /kaggle/working/vimind_4.0_pro_final


In [ ]:
# 9. [GIAI ĐOẠN 4: AGENTIC REINFORCEMENT LEARNING (GRPO 4.0)]
# Huấn luyện khả năng sử dụng công cụ Toán học, Thời tiết, Thời gian và suy nghĩ logic
import os
base_rl_model = 'out/dpo/vimind_4.0_dpo_final' if os.path.exists('out/dpo/vimind_4.0_dpo_final') else ('out/dpo/vimind_4.0_dpo.pth' if os.path.exists('out/dpo/vimind_4.0_dpo.pth') else ('out/sft_moe/vimind_4.0_moe_final' if os.path.exists('out/sft_moe/vimind_4.0_moe_final') else 'out/sft_moe'))
!python -u trainer/train_agent.py \
    --model_path {base_rl_model} \
    --save_dir out/agent_rl \
    --save_weight vimind_4.0_agent \
    --epochs 1 \
    --batch_size 2 \
    --num_rollouts 4 \
    --learning_rate 1e-5 \
    --fp16


In [ ]:
# 10. [GIAI ĐOẠN 5: BENCHMARK ĐÁNH GIÁ NĂNG LỰC GỌI CÔNG CỤ (TOOL CALL)]
import os
bench_model = 'out/agent_rl/vimind_4.0_agent_final' if os.path.exists('out/agent_rl/vimind_4.0_agent_final') else ('out/agent_rl/vimind_4.0_agent.pth' if os.path.exists('out/agent_rl/vimind_4.0_agent.pth') else ('out/dpo/vimind_4.0_dpo_final' if os.path.exists('out/dpo/vimind_4.0_dpo_final') else 'out/sft_moe'))
!python scripts/eval_toolcall.py --model_path {bench_model}


In [ ]:
# 11. [GIAI ĐOẠN 6: ĐÓNG GÓI SAFETENSORS VIMIND 4.0 NATIVE]
# Xuất xưởng file chuẩn model.safetensors cho ViMind 4.0 Native
import os
candidates = [
    'out/sft_moe/vimind_4.0_moe_final',
    'out/sft_moe/vimind_4.0_moe.pth',
    'out/pretrain/vimind_4.0_base_final',
    'out/pretrain/vimind_4.0_base.pth',
]
source_weight = next((c for c in candidates if os.path.exists(c)), 'out/sft_moe')
print(f'📦 Đóng gói ViMind 4.0 Native từ nguồn: {source_weight}')
!python scripts/convert_model.py \
    --input {source_weight} \
    --output /kaggle/working/vimind_4.0_moe_final \
    --model_size 64m \
    --moe \
    --num_experts 4 \
    --num_experts_per_tok 2

!ls -lh /kaggle/working/vimind_4.0_moe_final


In [ ]:
# 12. [GIAI ĐOẠN 7: ĐỐI ĐẦU TRỰC TIẾP (HEAD-TO-HEAD BATTLE) - VIMIND NATIVE VS PRO]
# So tài độ thông minh thực sự trên 6 câu hỏi cốt lõi
import os, json, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from model.model import ViMindConfig, ViMindForCausalLM
from safetensors.torch import load_file

test_cases = [
    'Xin chào, bạn là mô hình AI nào và bạn có khả năng gì nổi bật?',
    'Thủ đô của nước Cộng hòa Xã hội Chủ nghĩa Việt Nam là gì?',
    'Việt Nam có bao nhiêu tỉnh thành? Hãy kể tên một số tỉnh miền Trung.',
    'Tính giúp tôi kết quả của 25 * 18 + 750 / 5.',
    'Thời tiết hôm nay tại Đà Nẵng thế nào, có mát mẻ không?',
    'Vì sao ban ngày trời sáng còn ban đêm trời tối?'
]

print('=' * 80)
print('        🥊 ĐỐI ĐẦU TRỰC TIẾP: VIMIND 4.0 NATIVE vs VIMIND 4.0 PRO')
print('=' * 80)

# ==========================================
# 1. TEST VIMIND 4.0 NATIVE (MoE 198M/64M)
# ==========================================
print('\n🔵 [PHẦN 1] THỰC THI KIỂM THỬ TRÊN VIMIND 4.0 NATIVE (MoE 198M)')
native_dir = '/kaggle/working/vimind_4.0_moe_final'
if not os.path.exists(os.path.join(native_dir, 'model.safetensors')):
    native_dir = 'out/sft_moe/vimind_4.0_moe_final'

tok_native = AutoTokenizer.from_pretrained(native_dir if os.path.exists(os.path.join(native_dir, 'tokenizer.json')) else 'model')
cfg_native = ViMindConfig.from_pretrained(native_dir) if (os.path.isdir(native_dir) and os.path.exists(os.path.join(native_dir, 'config.json'))) else ViMindConfig(vocab_size=len(tok_native), hidden_size=640, num_hidden_layers=12, num_attention_heads=10, num_key_value_heads=5, intermediate_size=1728, use_moe=True, num_experts=4, num_experts_per_tok=2)
model_native = ViMindForCausalLM(cfg_native).cuda()

sf_path = os.path.join(native_dir, 'model.safetensors')
if os.path.exists(sf_path):
    model_native.load_state_dict(load_file(sf_path), strict=False)
    print(f'✅ Đã nạp thành công trọng số Native từ {sf_path}!')
elif os.path.exists('out/sft_moe/vimind_4.0_moe.pth'):
    st = torch.load('out/sft_moe/vimind_4.0_moe.pth', map_location='cuda')
    if 'model' in st: st = st['model']
    model_native.load_state_dict({k.replace('module.', ''): v for k, v in st.items()}, strict=False)
    print('✅ Đã nạp trọng số dự phòng Native .pth!')
model_native.eval()

for q in test_cases:
    prompt = tok_native.apply_chat_template([{'role': 'user', 'content': q}], tokenize=False, add_generation_prompt=True)
    inputs = tok_native(prompt, return_tensors='pt').input_ids.cuda()
    with torch.no_grad():
        out = model_native.generate(inputs, max_new_tokens=256, temperature=0.2, top_p=0.9, repetition_penalty=1.08, no_repeat_ngram_size=0, eos_token_id=tok_native.eos_token_id)
    reply = tok_native.decode(out[0][inputs.shape[1]:], skip_special_tokens=False)
    print(f'\n👤 Người dùng: {q}')
    print(f'🤖 [ViMind 4.0 Native]:\n{reply.strip()}')
    print('-' * 70)

# ==========================================
# 2. TEST VIMIND 4.0 PRO (0.5B Foundation)
# ==========================================
pro_dir = '/kaggle/working/vimind_4.0_pro_final'
if os.path.exists(pro_dir):
    print('\n\n🔴 [PHẦN 2] THỰC THI KIỂM THỬ TRÊN VIMIND 4.0 PRO (0.5B Foundation)')
    try:
        tok_pro = AutoTokenizer.from_pretrained(pro_dir)
        model_pro = AutoModelForCausalLM.from_pretrained(pro_dir, torch_dtype=torch.float16, device_map='auto')
        model_pro.eval()

        for q in test_cases:
            prompt = tok_pro.apply_chat_template([{'role': 'user', 'content': q}], tokenize=False, add_generation_prompt=True)
            inputs = tok_pro(prompt, return_tensors='pt').to('cuda')
            with torch.no_grad():
                out = model_pro.generate(**inputs, max_new_tokens=256, temperature=0.3, top_p=0.85, repetition_penalty=1.08, eos_token_id=tok_pro.eos_token_id)
            reply = tok_pro.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=False)
            print(f'\n👤 Người dùng: {q}')
            print(f'⚡ [ViMind 4.0 Pro]:\n{reply.strip()}')
            print('-' * 70)
    except Exception as e:
        print(f'⚠️ Kiểm thử Pro gặp ngoại lệ: {e}')
